# HotpotQA — Raw Data Exploration

## 1. Load dataset

In [2]:
import json
from pathlib import Path
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt

ROOT        = Path('.').resolve().parent
DATA_PATH   = ROOT / 'data' / 'hotpotqa' / 'validation.jsonl'
RESULTS_DIR = ROOT / 'results'

In [3]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [4]:
records = [json.loads(l) for l in DATA_PATH.read_text().splitlines() if l.strip()]
df = pd.DataFrame(records)
print(f'Loaded {len(df):,} validation questions from {DATA_PATH}')
df.head(3)

Loaded 7,405 validation questions from /Users/bercaakbayir/Desktop/projects/unimi-dse-nlp-assignment/data/hotpotqa/validation.jsonl


id  \
0  5a8b57f25542995d1e6f1371   
1  5a8c7595554299585d9e36b6   
2  5a85ea095542994775f606a8   

                                                                                                                                                  question  \
0                                                                                               Were Scott Derrickson and Ed Wood of the same nationality?   
1                                                   What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?   
2  What science fantasy young adult series, told in first person, has a set of companion books narrating the stories of enslaved worlds and alien species?   

              answer        type level  \
0                yes  comparison  hard   
1  Chief of Protocol      bridge  hard   
2          Animorphs      bridge  hard   

                                                                                                                                           supporting_facts  \
0                                                                                             {'title': ['Scott Derrickson', 'Ed Wood'], 'sent_id': [0, 0]}   
1                                                        {'title': ['Kiss and Tell (1945 film)', 'Shirley Temple', 'Shirley Temple'], 'sent_id': [0, 0, 1]}   
2  {'title': ['The Hork-Bajir Chronicles', 'The Hork-Bajir Chronicles', 'The Hork-Bajir Chronicles', 'Animorphs', 'Animorphs'], 'sent_id': [0, 1, 2, 0, 1]}   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

In [5]:
print(f'Shape  : {df.shape}')
print(f'Columns: {list(df.columns)}')

Shape  : (7405, 7)
Columns: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context']


## 2. Schema

In [6]:
df.dtypes

id                     str
question               str
answer                 str
type                   str
level                  str
supporting_facts    object
context             object
dtype: object

In [7]:
ex = records[1]
print('id              :', ex['id'])
print('question        :', ex['question'])
print('answer          :', ex['answer'])
print('type            :', ex['type'])
print('level           :', ex['level'])
print('supporting_facts:', ex['supporting_facts'])
print()
print('context titles  :', ex['context']['title'])
print('context[0] sents:', ex['context']['sentences'][0][:2])

id              : 5a8c7595554299585d9e36b6
question        : What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?
answer          : Chief of Protocol
type            : bridge
level           : hard
supporting_facts: {'title': ['Kiss and Tell (1945 film)', 'Shirley Temple', 'Shirley Temple'], 'sent_id': [0, 0, 1]}

context titles  : ['Meet Corliss Archer', 'Shirley Temple', 'Janet Waldo', 'Meet Corliss Archer (TV series)', 'Lord High Treasurer', 'A Kiss for Corliss', 'Kiss and Tell (1945 film)', 'Secretary of State for Constitutional Affairs', 'Village accountant', 'Charles Craft']
context[0] sents: ["Meet Corliss Archer, a program from radio's Golden Age, ran from January 7, 1943 to September 30, 1956.", ' Although it was CBS\'s answer to NBC\'s popular "A Date with Judy", it was also broadcast by NBC in 1948 as a summer replacement for "The Bob Hope Show".']


## 3. Basic statistics

In [8]:
print('=== Question type distribution ===')
print(df['type'].value_counts())
print()
print('=== Difficulty level distribution ===')
print(df['level'].value_counts())

=== Question type distribution ===
type
bridge        5918
comparison    1487
Name: count, dtype: int64

=== Difficulty level distribution ===
level
hard    7405
Name: count, dtype: int64


In [9]:
5918

5918

In [10]:
df['q_len'] = df['question'].str.len()
df['a_len'] = df['answer'].str.len()

print('=== Question length (chars) ===')
print(df['q_len'].describe().round(1))
print()
print('=== Answer length (chars) ===')
print(df['a_len'].describe().round(1))

=== Question length (chars) ===
count    7405.0
mean       92.2
std        32.6
min        32.0
25%        69.0
50%        86.0
75%       109.0
max       288.0
Name: q_len, dtype: float64

=== Answer length (chars) ===
count    7405.0
mean       15.4
std        12.0
min         1.0
25%         8.0
50%        13.0
75%        20.0
max       166.0
Name: a_len, dtype: float64


In [11]:
df['n_supporting'] = df['supporting_facts'].apply(lambda x: len(x['title']))
print('=== Supporting facts count per question ===')
print(df['n_supporting'].value_counts().sort_index())

=== Supporting facts count per question ===
n_supporting
2    4990
3    1774
4     537
5      80
6      14
7       9
8       1
Name: count, dtype: int64


In [12]:
df['n_context'] = df['context'].apply(lambda x: len(x['title']))
print('=== Context articles per question (distractor setting = always 10) ===')
print(df['n_context'].value_counts())

=== Context articles per question (distractor setting = always 10) ===
n_context
10    7345
2       21
4        9
6        8
3        7
5        6
8        4
9        3
7        2
Name: count, dtype: int64


## 4. Flatten context into passage-level rows

In [13]:
passage_rows = []
for rec in records:
    for title, sents in zip(rec['context']['title'], rec['context']['sentences']):
        passage_rows.append({
            'qid':   rec['id'],
            'title': title,
            'text':  ' '.join(sents),
        })

passages_df = pd.DataFrame(passage_rows)
passages_df['text_len'] = passages_df['text'].str.len()
print(f'Total passages: {len(passages_df):,}')
print()
print('=== Passage text length (chars) ===')
print(passages_df['text_len'].describe().round(1))
passages_df.head(5)

Total passages: 73,700

=== Passage text length (chars) ===
count    73700.0
mean       552.7
std        344.7
min         29.0
25%        329.0
50%        489.0
75%        696.0
max       8268.0
Name: text_len, dtype: float64


,qid,title,text,text_len
0,5a8b57f25542995d1e6f1371,Ed Wood (film),"Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood. The film concerns the period in Wood's life when he made his best-known films as well as his relationship with actor Bela Lugosi, played by Martin Landau. Sarah Jessica Parker, Patricia Arquette, Jeffrey Jones, Lisa Marie, and Bill Murray are among the supporting cast.",425
1,5a8b57f25542995d1e6f1371,Scott Derrickson,"Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer. He lives in Los Angeles, California. He is best known for directing horror films such as ""Sinister"", ""The Exorcism of Emily Rose"", and ""Deliver Us From Evil"", as well as the 2016 Marvel Cinematic Universe installment, ""Doctor Strange.""",327
2,5a8b57f25542995d1e6f1371,"Woodson, Arkansas","Woodson is a census-designated place (CDP) in Pulaski County, Arkansas, in the United States. Its population was 403 at the 2010 census. It is part of the Little Rock–North Little Rock–Conway Metropolitan Statistical Area. Woodson and its accompanying Woodson Lake and Wood Hollow are the namesake for Ed Wood Sr., a prominent plantation owner, trader, and businessman at the turn of the 20th century. Woodson is adjacent to the Wood Plantation, the largest of the plantations own by Ed Wood Sr.",499
3,5a8b57f25542995d1e6f1371,Tyler Bates,"Tyler Bates (born June 5, 1965) is an American musician, music producer, and composer for films, television, and video games. Much of his work is in the action and horror film genres, with films like ""Dawn of the Dead, 300, Sucker Punch,"" and ""John Wick."" He has collaborated with directors like Zack Snyder, Rob Zombie, Neil Marshall, William Friedkin, Scott Derrickson, and James Gunn. With Gunn, he has scored every one of the director's films; including ""Guardians of the Galaxy"", which became one of the highest grossing domestic movies of 2014, and its 2017 sequel. In addition, he is also the lead guitarist of the American rock band Marilyn Manson, and produced its albums ""The Pale Emperor"" and ""Heaven Upside Down"".",729
4,5a8b57f25542995d1e6f1371,Ed Wood,"Edward Davis Wood Jr. (October 10, 1924 – December 10, 1978) was an American filmmaker, actor, writer, producer, and director.",126


In [14]:
passages_df["text_len"] = passages_df["text"].str.len()
print("=== Passage text length (chars) ===")
print(passages_df["text_len"].describe().round(1))

=== Passage text length (chars) ===
count    73700.0
mean       552.7
std        344.7
min         29.0
25%        329.0
50%        489.0
75%        696.0
max       8268.0
Name: text_len, dtype: float64


## 5. Sample questions by type

In [15]:
print('=== Comparison questions (sample) ===')
for _, r in df[df['type'] == 'comparison'][['question', 'answer']].sample(5, random_state=42).iterrows():
    print(f'  Q: {r["question"]}')
    print(f'  A: {r["answer"]}')
    print()

=== Comparison questions (sample) ===
  Q: Which opera has more acts, La jolie fille de Perth or Mitridate, re di Ponto?
  A: La jolie fille de Perth "(The Fair Maid of Perth)" is an opera in four acts

  Q: Which composer was a French Romantic composer in the 1800's, Hector Berlioz or Gaetano Donizetti?
  A: Louis-Hector Berlioz

  Q: What American filmmaker was the co-founder of Troma Entertainment film studio, Stanley Lloyd Kaufman, Jr. or Jay T. Wright?
  A: Stanley Lloyd Kaufman, Jr.

  Q: which plant has the highest specie  Panicum or Populus 
  A: Panicum

  Q: Which facility was founded in Missouri, Discovery Zone or Valentino's?
  A: Discovery Zone



In [16]:
print('=== Bridge questions (sample) ===')
for _, r in df[df['type'] == 'bridge'][['question', 'answer']].sample(5, random_state=42).iterrows():
    print(f'  Q: {r["question"]}')
    print(f'  A: {r["answer"]}')
    print()

=== Bridge questions (sample) ===
  Q: What Mexican actress of film, television, and theatre was a protagonist in La Heredera?
  A: Silvia Navarro

  Q: Grown-Ups starred the actor who was best known for which role on  "'Allo 'Allo!"?
  A: Captain Hans Geering

  Q: Li Yitong made her television debut on which network?
  A: Dragon TV

  Q: Ross Pople worked with this French dominant figure of the post-war classical music world.
  A: Pierre Louis Joseph Boulez

  Q: How many students were enrolled in American professional bowler Chris Barnes' high school in the 2010-2011 school year?
  A: 1,840 students

